# Последовательность действий



```mermaid
sequenceDiagram
    participant Cash
    participant Pool
    participant Short

    Cash->>Pool: Создать
    Pool->>Short: Создать

    opt Упало до BEP
        Short->>Cash: Закрыть
        Pool->>Cash: Закрыть
    end
```

```mermaid
flowchart TD
    market([price])-->Pool
    market([price])-->Short
```

```mermaid
classDiagram 
direction LR
class Pool{
    capital
    p_b
    p_n
    p_a
    -calc_range()
    open()
    close()
    reinvest()
}

class Short{
    BEP
    p_b**
    open()
    close()
    is_close()
    get_size()
}

Pool--*Short
```

```mermaid
---
title: Триггер
---
stateDiagram-v2
    
    calc_amount : Рассчитать стоимость
    input_data : time, price
    state trigger <<choice>>
    
    [*] --> input_data
    input_data --> trigger 
    trigger --> close_short : price > p_b
    trigger --> close_all : price близко к BEP
    trigger --> calc_amount
    close_short --> calc_amount
    close_all --> calc_amount
    calc_amount --> write_metrics
    write_metrics --> [*]
```

```mermaid
---
title: Закрытие Short (close_short)
---
stateDiagram-v2
    [*] --> [*]
    %%open --> triggers
    %%triggers --> close
    %%close --> [*]
    %%triggers --> [*]

    %%if_open --> triggers : True

    %%state open {
    %%    calc_ranges : Рассчитать пределы
    %%}    
    
    %%state close {
    %%    to_capital : Обналичить
    %%}
```

```mermaid
---
title: Закрыть Всё (close_all)
---
stateDiagram-v2
    [*] --> close_short
    close_short --> close_pool
    close_pool --> [*]

```

### Загрузка данных из файла

In [2]:
import pandas as pd


data_file = "../eth_2025_full.csv"
df = pd.read_csv(data_file, header=None, skiprows=1)

#### Проверка времени на цикл перебора  

In [6]:
%time
for index,row in df.iterrows():
    # print(f"index: {index}")
    pass

CPU times: total: 0 ns
Wall time: 0 ns


##### Пред рассчитанные данные:
- Верхняя граница цены - $P_b$
- Нижняя граница цены - $P_a$
- Ликвидность - $L$
- Граница безубыточности - $BEP$

---

#### Просчеты на каждый шаг
##### Входные данные:
- Время - $t$
- Цена - $price$

---

##### Данные содержащиеся в $Poll$:
- Время пред идущего шага
- Цена на пред идущем шаге

##### Данные на выходе:
- Размер $Short$
- Размер $


- 

##### Размер Пул

$\text{размер пул} \equiv \text{капитал}$


```python
def calculate_pool_value(price, pa, pb, l):
    """Рассчитывает стоимость пула при заданной цене ТОЛЬКО по формуле V3"""
    if l <= 0:
        return 0.0
    sqrt_price = np.sqrt(price)
    sqrt_pa = np.sqrt(pa)
    sqrt_pb = np.sqrt(pb)
    if price <= pa:
        pool_eth = l * (1/sqrt_pa - 1/sqrt_pb)
        return pool_eth * price
    elif price >= pb:
        return l * (sqrt_pb - sqrt_pa)
    else:
        pool_usdc = l * (sqrt_price - sqrt_pa)
        pool_eth = l * (1/sqrt_price - 1/sqrt_pb)
        return pool_usdc + pool_eth * price
```

$$
PoolSize(price, p_a, p_b, l) = 
\begin{cases}
0 & l\leq 0 \\
l \cdot (\frac{1}{\sqrt{p_a}}-\frac{1}{\sqrt{p_b}}) \cdot price & price \leq p_a \\
l \cdot (\sqrt{p_b}-\sqrt{p_a}) & price \geq p_b\\
pool_{usdc} + pool_{eth} \cdot price & иначе
\end{cases}
$$

$$
pool_{usdc} = l \cdot (\sqrt{price}-\sqrt{p_a}) \\
pool_{eth} = l \cdot (\frac{1}{\sqrt{price}}-\frac{1}{\sqrt{p_b}})
$$

##### Размер Шорт

```python
def calculate_short_size(entry_price, pa, pb, l):
    """Рассчитывает размер шорт-позиции"""
    if entry_price < pa or entry_price > pb or l == 0:
        return 0.0
    sqrt_entry = np.sqrt(entry_price)
    sqrt_pa = np.sqrt(pa)
    return l * (1/sqrt_pa - 1/sqrt_entry)
```

$$
ShortSize(price, p_a, p_b, l) = 
\begin{cases}
0 & price < p_a \\
0 & price > p_b \\
0 & l = 0 \\
l \cdot (\frac{1}{\sqrt{p_a}}-\frac{1}{\sqrt{price}}) & иначе
\end{cases}
$$

##### Ликвидность

$L$ - Ликвидность

$Cap$ - Капитал

$$
L(Cap, p_n, p_a, p_b )= 
\begin{cases}
\frac{Cap}{t_1+t_2} & t_1 + t_2 \neq 0 \\
0 & otherwise \\
\end{cases}
$$
где
$$
t_1=\sqrt{p_n} - \sqrt{p_a} \\
t_2=(\sqrt{p_n} - \sqrt{p_a}) + p_n \cdot (\frac{1}{\sqrt{p_n}} - \frac{1}{\sqrt{p_b}}) \\
$$
где
$$
p_n=\begin{cases}
\frac{p_a + p_b}{2} & p_a \geq p_n \geq p_b \\
p_n & otherwise
\end{cases}
$$